# Gemma 实际生成复核：同一批图片、原问法与中性问法

只运行 Gemma-12B。

[实验设置与运行说明](../../../docs/experiment_notes/90_followup_visual_and_relation_use/244dffb77616f9ba.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready.')


In [ ]:
%%cell_progress 1/13 Mount Drive and locate project
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/13 Choose reviewed Gemma ZIP and run ID
RUN_ID = 'main'
SOURCE_ZIP = None  # Optional exact path to the reviewed Gemma frozen-validation ZIP.
DOWNLOAD_OUTPUT = True  # Drive copy is always retained; try browser download at the end.
# None checks its original Drive output, /content, and the project root only.
# This notebook ONLY runs Gemma-12B. Do not switch MODEL_KEY to Qwen.


In [ ]:
%%cell_progress 3/13 Install pinned inference dependencies
import importlib
import os

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'hf_xet>=1.1', 'tqdm>=4.66',
    'pandas>=2.2', 'matplotlib>=3.8', 'Pillow==11.3.0',
])

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit source:', toolkit_path)


In [ ]:
%%cell_progress 4/13 Load fixed response grammar and audit helpers
import types, sys
_parent=types.ModuleType("visual_information_support_core")
sys.modules[_parent.__name__]=_parent
exec(compile('"""CPU design, rendering, prompts and family-paired analysis for information support."""\nfrom __future__ import annotations\n\nimport hashlib\nimport itertools\nimport json\nfrom pathlib import Path\nfrom collections import Counter\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageDraw, ImageFont\n\nSCHEMA = \'visual-information-support-v1\'\nCONDITIONS = (\'image\', \'image_weight\', \'image_distance\', \'image_full\', \'text_full\', \'legacy_image\')\nSTRATA = (\'congruent\', \'balanced\', \'equal_weight_distance\', \'weight_conflict\', \'binding_swap\')\nLABELS = (\'A\', \'B\', \'C\')\nSEMANTICS = (\'left\', \'balanced\', \'right\')\nSEMANTIC_TEXT = {\'left\': \'left side\', \'balanced\': \'equal\', \'right\': \'right side\'}\nRULE_TEXT = (\'Each square is one unit of weight. Distance is measured in tick \'\n             \'intervals from the center pivot. Torque equals weight times distance. \'\n             \'Total torque on a side is the sum over stacks on that side.\')\nREPLY_TEXT = (\'Do not explain your reasoning. Reply with exactly one option letter \'\n              \'and nothing else. Valid replies are: A, B, C.\')\nMODEL_SPECS = {\n    \'gemma12b\': dict(model_id=\'google/gemma-4-12B-it\',\n                    revision=\'707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7\'),\n    \'qwen27b\': dict(model_id=\'Qwen/Qwen3.5-27B\',\n                   revision=\'373612cb4f9288fcf550b526d86d85febedd9801\'),\n}\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,\n                                    allow_nan=False).encode()).hexdigest()\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef save_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False) + \'\\n\', encoding=\'utf-8\')\n    tmp.replace(path)\n\n\ndef freeze(path, obj):\n    if Path(path).exists():\n        assert read_json(path) == obj, f\'Frozen configuration changed: {path}. Use a new RUN_ID.\'\n    else:\n        save_json(path, obj)\n    return digest(obj)\n\n\ndef signature(stacks):\n    direct = tuple(sorted((s[\'side\'], int(s[\'distance\']), int(s[\'units\'])) for s in stacks))\n    mirror = tuple(sorted((\'right\' if s == \'left\' else \'left\', d, w) for s, d, w in direct))\n    return min(direct, mirror)\n\n\ndef geometries_in(obj):\n    found = set()\n    if isinstance(obj, dict):\n        if isinstance(obj.get(\'stacks\'), list) and obj[\'stacks\']:\n            stacks = obj[\'stacks\']\n            if all(isinstance(s, dict) and {\'side\', \'distance\', \'units\'} <= s.keys() for s in stacks):\n                found.add(signature(stacks))\n        for key, value in obj.items():\n            if key == \'signatures\' and isinstance(value, list):\n                for sig in value:\n                    found.add(signature([dict(side=s, distance=d, units=w) for s, d, w in sig]))\n            elif key != \'stacks\':\n                found.update(geometries_in(value))\n    elif isinstance(obj, list):\n        for item in obj:\n            found.update(geometries_in(item))\n    return found\n\n\ndef load_history(project, run_root, source_run=\'main\', extra_files=(), progress=print):\n    """Read only explicit old supports/snapshots and shallow dataset manifests; no recursive Drive walk."""\n    project, run_root = Path(project), Path(run_root)\n    lock = run_root / \'historical_exclusion.json\'\n    if lock.exists():\n        old = read_json(lock)\n        assert old[\'source_run\'] == source_run and old[\'extra_files\'] == [str(p) for p in extra_files]\n        for name, sha in old[\'files\'].items():\n            assert file_hash(name) == sha, \'Historical input changed: \' + name\n        progress(f"Reusing frozen exclusions: {len(old[\'signatures\'])} geometries; no directory scan.")\n        return geometries_in(old)\n    paths = set()\n    data = project / \'data\'\n    if data.exists():\n        paths.update(data.glob(\'*/scenes.jsonl\'))\n        if (data / \'scenes.jsonl\').is_file():\n            paths.add(data / \'scenes.jsonl\')\n    results = project / \'results\'\n    if results.exists():\n        # Only top-level experiment directories and this known run, never activation shards.\n        for experiment in results.iterdir():\n            if experiment.is_dir() and experiment.resolve() != run_root.parent.resolve():\n                for folder in (experiment, experiment / source_run):\n                    for name in (\'historical_geometry_snapshot.json\', \'support.json\', \'scenes.jsonl\'):\n                        if (folder / name).is_file():\n                            paths.add(folder / name)\n    for name in extra_files:\n        p = Path(name)\n        assert p.is_file(), f\'EXTRA_HISTORY_FILES must name files: {p}\'\n        paths.add(p)\n    assert paths, \'No historical manifests found. Restore old support/snapshot files, or set EXTRA_HISTORY_FILES.\'\n    signatures, hashes = set(), {}\n    for i, path in enumerate(sorted(paths)):\n        progress(f\'Read historical manifest {i+1}/{len(paths)}: {path.name}\')\n        h = hashlib.sha256()\n        with path.open(\'rb\') as stream:\n            if path.suffix == \'.jsonl\':\n                for line in stream:\n                    h.update(line)\n                    if line.strip():\n                        signatures.update(geometries_in(json.loads(line.decode(\'utf-8-sig\'))))\n            else:\n                raw = stream.read()\n                h.update(raw)\n                signatures.update(geometries_in(json.loads(raw.decode(\'utf-8-sig\'))))\n        hashes[str(path)] = h.hexdigest()\n    assert signatures, \'No usable historical geometry was found in the selected manifests.\'\n    save_json(lock, dict(source_run=source_run, extra_files=[str(p) for p in extra_files],\n                         files=hashes, signatures=[[list(t) for t in sig] for sig in sorted(signatures)],\n                         scope=\'Listed manifests/snapshots only; unseen ZIPs or unlisted runs are not certified disjoint.\'))\n    return signatures\n\n\ndef scene_features(stacks):\n    w = {s: sum(x[\'units\'] for x in stacks if x[\'side\'] == s) for s in (\'left\', \'right\')}\n    torque = {s: sum(x[\'units\'] * x[\'distance\'] for x in stacks if x[\'side\'] == s)\n              for s in (\'left\', \'right\')}\n    delta = torque[\'left\'] - torque[\'right\']\n    weight = w[\'left\'] - w[\'right\']\n    outcome = lambda x: \'left\' if x > 0 else \'right\' if x < 0 else \'balanced\'\n    return dict(left_torque=torque[\'left\'], right_torque=torque[\'right\'], net_torque=delta,\n                weight_score=weight, outcome=outcome(delta), weight_outcome=outcome(weight))\n\n\ndef allocation(n):\n    assert isinstance(n, int) and n >= 32 and n % 8 == 0\n    return dict(congruent=n//8, balanced=n//8, equal_weight_distance=n//4,\n                weight_conflict=n//4, binding_swap=n//4)\n\n\ndef _candidate(rng, stratum):\n    dl, dr = [sorted(rng.choice(np.arange(1, 7), 2, replace=False).tolist()) for _ in range(2)]\n    wl, wr = rng.integers(1, 7, size=(2, 2)).tolist()\n    if stratum == \'equal_weight_distance\':\n        wr = wl.copy()\n        if not (all(a > b for a, b in zip(dl, dr)) or all(a < b for a, b in zip(dl, dr))):\n            return None\n    stacks = [dict(side=side, distance=d, units=w) for side, distances, weights in\n              [(\'left\', dl, wl), (\'right\', dr, wr)] for d, w in zip(distances, weights)]\n    f = scene_features(stacks)\n    if stratum == \'congruent\':\n        distance_delta = sum(dl) - sum(dr)\n        if not (f[\'weight_score\'] * f[\'net_torque\'] > 0 and distance_delta * f[\'net_torque\'] > 0):\n            return None\n    elif stratum == \'balanced\' and f[\'net_torque\'] != 0:\n        return None\n    elif stratum == \'weight_conflict\' and f[\'weight_score\'] * f[\'net_torque\'] >= 0:\n        return None\n    roles = {\'base\': stacks}\n    if stratum == \'binding_swap\':\n        swapped = [dict(s) for s in stacks]\n        swapped[0][\'units\'], swapped[1][\'units\'] = swapped[1][\'units\'], swapped[0][\'units\']\n        if f[\'net_torque\'] * scene_features(swapped)[\'net_torque\'] >= 0:\n            return None\n        roles[\'swapped\'] = swapped\n    return roles\n\n\ndef generate_families(n, seed, excluded, phase):\n    rng = np.random.default_rng(seed)\n    used = set(excluded)\n    families = []\n    for stratum, count in allocation(n).items():\n        accepted = 0\n        for attempt in range(300000):\n            roles = _candidate(rng, stratum)\n            if roles is None:\n                continue\n            sigs = {signature(s) for s in roles.values()}\n            if len(sigs) != len(roles) or sigs & used:\n                continue\n            fid = \'vis_\' + digest([SCHEMA, phase, stratum, sorted(sigs)])[:16]\n            slot_ids = np.random.default_rng(int(digest([fid, \'ids\'])[:16], 16)).permutation(4).tolist()\n            records = []\n            for role, stacks in roles.items():\n                for mirror in (False, True):\n                    actual = [dict(s, stack_id=f\'S{slot_ids[i]+1}\',\n                                   side=(\'right\' if s[\'side\'] == \'left\' else \'left\') if mirror else s[\'side\'])\n                              for i, s in enumerate(stacks)]\n                    sid = f\'{fid}__{role}__m{int(mirror)}\'\n                    records.append(dict(scene_id=sid, role=role, mirror=mirror,\n                                        stacks=actual, features=scene_features(actual)))\n            families.append(dict(family_id=fid, stratum=stratum, phase=phase, scenes=records))\n            used.update(sigs)\n            accepted += 1\n            if accepted == count:\n                break\n        assert accepted == count, f\'Insufficient unused {stratum} geometries: {accepted}/{count}; do not relax exclusions.\'\n    families.sort(key=lambda f: digest([seed, f[\'family_id\']]))\n    validate_families(families, excluded)\n    return families\n\n\ndef validate_families(families, excluded=()):\n    used = set(excluded)\n    assert len({f[\'family_id\'] for f in families}) == len(families)\n    for family in families:\n        base = [s for s in family[\'scenes\'] if not s[\'mirror\']]\n        sigs = {signature(s[\'stacks\']) for s in base}\n        assert len(sigs) == len(base) and not sigs & used\n        used.update(sigs)\n        for scene in family[\'scenes\']:\n            assert scene_features(scene[\'stacks\']) == scene[\'features\']\n            assert len(scene[\'stacks\']) == 4\n            assert len({(s[\'side\'], s[\'distance\']) for s in scene[\'stacks\']}) == 4\n            assert len({s[\'stack_id\'] for s in scene[\'stacks\']}) == 4\n            assert all(1 <= s[\'units\'] <= 6 and 1 <= s[\'distance\'] <= 6 for s in scene[\'stacks\'])\n        if family[\'stratum\'] == \'binding_swap\':\n            a, b = base\n            assert a[\'features\'][\'net_torque\'] * b[\'features\'][\'net_torque\'] < 0\n            for side in (\'left\', \'right\'):\n                aa = [s for s in a[\'stacks\'] if s[\'side\'] == side]\n                bb = [s for s in b[\'stacks\'] if s[\'side\'] == side]\n                assert sorted(s[\'units\'] for s in aa) == sorted(s[\'units\'] for s in bb)\n                assert sorted(s[\'distance\'] for s in aa) == sorted(s[\'distance\'] for s in bb)\n            assert {s[\'stack_id\']: (s[\'side\'], s[\'distance\']) for s in a[\'stacks\']} == {\n                s[\'stack_id\']: (s[\'side\'], s[\'distance\']) for s in b[\'stacks\']}\n\n\ndef mappings_for(family_id):\n    offset = int(digest([family_id, \'mapping\'])[:8], 16) % 3\n    return [tuple(SEMANTICS[(i + k + offset) % 3] for i in range(3)) for k in range(3)]\n\n\ndef public_fields(scene, condition):\n    assert condition in CONDITIONS\n    full = condition in (\'image_full\', \'text_full\')\n    return [dict(stack_id=s[\'stack_id\'], side=s[\'side\'],\n                 weight=s[\'units\'] if full or condition == \'image_weight\' else \'not provided\',\n                 distance=s[\'distance\'] if full or condition == \'image_distance\' else \'not provided\')\n            for s in sorted(scene[\'stacks\'], key=lambda s: s[\'stack_id\'])]\n\n\ndef prompt_for(scene, condition, semantics):\n    options = \'\\n\'.join(f\'{label}. {SEMANTIC_TEXT[meaning]}\' for label, meaning in zip(LABELS, semantics))\n    if condition == \'legacy_image\':\n        parts = [\'Answer this question using only the supplied balance-scale scene.\', RULE_TEXT]\n    else:\n        fields = public_fields(scene, condition)\n        table = \'\\n\'.join(f"{r[\'stack_id\']}: side={r[\'side\']}; weight={r[\'weight\']}; distance={r[\'distance\']}" for r in fields)\n        parts = [\'Answer this question using the supplied scene information.\', RULE_TEXT,\n                 \'Stack IDs identify positions; they do not encode weight or distance. \'\n                 \'Use any supplied image and provided values. A field marked not provided supplies no numeric value.\',\n                 \'Provided stack information:\\n\' + table]\n    parts += [\'Question: Which way will the balance scale tip?\', \'Options:\\n\' + options, REPLY_TEXT]\n    return \'\\n\\n\'.join(parts)\n\n\ndef trial_specs(families):\n    for family in families:\n        for scene in family[\'scenes\']:\n            for mapping_index, semantics in enumerate(mappings_for(family[\'family_id\'])):\n                for condition in CONDITIONS:\n                    eid = scene[\'scene_id\'] + f\'__map{mapping_index}__{condition}\'\n                    yield dict(trial_id=eid, family_id=family[\'family_id\'], stratum=family[\'stratum\'],\n                               scene_id=scene[\'scene_id\'], role=scene[\'role\'], mirror=scene[\'mirror\'],\n                               mapping_index=mapping_index, semantics=list(semantics), condition=condition,\n                               gold_semantic=scene[\'features\'][\'outcome\'],\n                               gold_label=LABELS[semantics.index(scene[\'features\'][\'outcome\'])],\n                               weight_outcome=scene[\'features\'][\'weight_outcome\'])\n\n\ndef render_images(families, image_dir):\n    from visual_torque_toolkit.render import RenderStyle, render_scene\n    from visual_torque_toolkit.world import Scene, Side, Stack\n    image_dir = Path(image_dir)\n    image_dir.mkdir(parents=True, exist_ok=True)\n    lock = image_dir.parent / \'image_manifest.json\'\n    if lock.exists():\n        manifest = read_json(lock)\n        expected = {s[\'scene_id\'] + suffix for f in families for s in f[\'scenes\']\n                    for suffix in (\'__plain.png\', \'__ids.png\')}\n        assert set(manifest) == expected\n        for name, entry in manifest.items():\n            assert file_hash(image_dir / name) == entry[\'sha256\'], \'Frozen image missing/changed: \' + name\n        return manifest\n    style = RenderStyle()\n    # Built-in scalable Pillow font travels with the pinned Pillow package.\n    font = ImageFont.load_default(size=18)\n    manifest = {}\n    for family in families:\n        for record in family[\'scenes\']:\n            scene = Scene(record[\'scene_id\'], tuple(Stack(s[\'stack_id\'], Side(s[\'side\']), s[\'distance\'], s[\'units\'])\n                                                   for s in record[\'stacks\']))\n            plain = image_dir / (record[\'scene_id\'] + \'__plain.png\')\n            render_scene(scene, plain, style)\n            with Image.open(plain) as source:\n                labeled = source.convert(\'RGB\')\n            draw = ImageDraw.Draw(labeled)\n            spacing = int(style.width * .38) / style.ticks_per_side\n            boxes = []\n            for s in scene.stacks:\n                x = round(style.width // 2 + (-1 if s.side is Side.LEFT else 1) * s.distance * spacing)\n                y = style.beam_y + 25\n                box = draw.textbbox((x, y), s.stack_id, font=font, anchor=\'mt\')\n                assert 0 <= box[0] < box[2] < style.width and 0 <= box[1] < box[3] < style.height\n                assert all(box[2] <= b[0] or box[0] >= b[2] or box[3] <= b[1] or box[1] >= b[3] for b in boxes)\n                boxes.append(box)\n                draw.text((x, y), s.stack_id, font=font, anchor=\'mt\', fill=style.ink)\n            labeled.save(image_dir / (record[\'scene_id\'] + \'__ids.png\'))\n            for name in (plain.name, record[\'scene_id\'] + \'__ids.png\'):\n                manifest[name] = dict(sha256=file_hash(image_dir / name), width=style.width, height=style.height)\n    save_json(lock, manifest)\n    return manifest\n\n\ndef generation_audit_ids(families):\n    selected = set()\n    for stratum in STRATA:\n        choices = sorted((f for f in families if f[\'stratum\'] == stratum), key=lambda f: digest([\'generation\', f[\'family_id\']]))[:2]\n        for f in choices:\n            scene = next(s for s in f[\'scenes\'] if s[\'role\'] == \'base\' and not s[\'mirror\'])\n            k = int(digest([\'audit-map\', f[\'family_id\']])[:8], 16) % 3\n            selected.update(scene[\'scene_id\'] + f\'__map{k}__{c}\' for c in CONDITIONS)\n    return selected\n\n\ndef strict_label(text):\n    value = text.strip()\n    return value if value in LABELS else None\n\n\ndef holm(p_values):\n    p = np.asarray(p_values, dtype=float)\n    order = np.argsort(p)\n    adjusted = np.empty(len(p))\n    previous = 0.\n    for rank, i in enumerate(order):\n        previous = max(previous, min(1., p[i] * (len(p) - rank)))\n        adjusted[i] = previous\n    return adjusted\n\n\ndef paired_stats(values, key, replicates=10000):\n    x = np.asarray(values, dtype=float)\n    assert x.ndim == 1 and len(x) and np.isfinite(x).all()\n    rng = np.random.default_rng(int(digest(key)[:16], 16))\n    draws = np.concatenate([rng.choice(x, (min(250, replicates-i), len(x)), replace=True).mean(axis=1)\n                            for i in range(0, replicates, 250)])\n    nonzero = x[x != 0]\n    observed = abs(float(x.sum()))\n    if len(nonzero) <= 16:\n        permuted = np.array([sum(a*b for a, b in zip(signs, nonzero))\n                             for signs in itertools.product((-1., 1.), repeat=len(nonzero))])\n        p = float(np.mean(np.abs(permuted) >= observed - 1e-12))\n    else:\n        extreme = 0\n        for i in range(0, replicates, 250):\n            signs = rng.choice((-1., 1.), size=(min(250, replicates-i), len(nonzero)))\n            extreme += int((np.abs(signs @ nonzero) >= observed - 1e-12).sum())\n        p = (extreme + 1) / (replicates + 1)\n    return dict(n_families=int(len(x)), mean=float(x.mean()),\n                lower=float(np.quantile(draws, .025)), upper=float(np.quantile(draws, .975)), p_value=p)\n\n\ndef analyze_rows(rows, families, outdir, model_key, phase, replicates=10000):\n    outdir = Path(outdir)\n    outdir.mkdir(parents=True, exist_ok=True)\n    df = pd.DataFrame(rows)\n    expected = {s[\'trial_id\'] for s in trial_specs(families)}\n    assert df.trial_id.is_unique and set(df.trial_id) == expected, \'Analysis requires every planned trial.\'\n    df.to_json(outdir / \'trial_results.jsonl\', orient=\'records\', lines=True, force_ascii=False)\n    metrics = [\'correct\', \'gold_margin\', \'predicted_balance\', \'wrong_follows_weight\']\n    fam = df.groupby([\'family_id\', \'stratum\', \'condition\'])[metrics].mean().reset_index()\n    fam.to_csv(outdir / \'family_metrics.csv\', index=False)\n    summaries = []\n    for scope in (\'all\', *STRATA):\n        subset = fam if scope == \'all\' else fam[fam.stratum == scope]\n        for condition, group in subset.groupby(\'condition\'):\n            row = dict(model=model_key, phase=phase, scope=scope, condition=condition,\n                       **paired_stats(group.correct, [model_key, phase, scope, condition], replicates))\n            row.pop(\'p_value\')\n            row.update(gold_margin=float(group.gold_margin.mean()),\n                       balance_answer_rate=float(group.predicted_balance.mean()),\n                       wrong_weight_response_rate=float(group.wrong_follows_weight.mean()))\n            summaries.append(row)\n    summary = pd.DataFrame(summaries)\n    summary.to_csv(outdir / \'accuracy_by_condition.csv\', index=False)\n    pair_rows = []\n    subset = df[df.stratum == \'binding_swap\']\n    for keys, group in subset.groupby([\'family_id\', \'condition\', \'mirror\', \'mapping_index\']):\n        assert set(group.role) == {\'base\', \'swapped\'} and len(group) == 2\n        a, b = [group[group.role == role].iloc[0] for role in (\'base\', \'swapped\')]\n        pair_rows.append(dict(family_id=keys[0], condition=keys[1], mirror=keys[2], mapping_index=keys[3],\n                              both_correct=int(a.correct and b.correct),\n                              answer_changed=int(a.predicted_semantic != b.predicted_semantic)))\n    pairs = pd.DataFrame(pair_rows)\n    pairs.to_csv(outdir / \'binding_pair_trials.csv\', index=False)\n    pair_fam = pairs.groupby([\'family_id\', \'condition\'])[[\'both_correct\', \'answer_changed\']].mean().reset_index()\n    pair_fam.to_csv(outdir / \'binding_pair_families.csv\', index=False)\n    definitions = [\n        (\'weight_support_all\', \'all\', [(\'image_weight\', 1), (\'image\', -1)], \'correct\'),\n        (\'distance_support_all\', \'all\', [(\'image_distance\', 1), (\'image\', -1)], \'correct\'),\n        (\'full_support_all\', \'all\', [(\'image_full\', 1), (\'image\', -1)], \'correct\'),\n        (\'joint_support_interaction\', \'all\', [(\'image_full\', 1), (\'image_weight\', -1), (\'image_distance\', -1), (\'image\', 1)], \'correct\'),\n        (\'distance_equal_weight\', \'equal_weight_distance\', [(\'image_distance\', 1), (\'image\', -1)], \'correct\'),\n        (\'full_weight_conflict\', \'weight_conflict\', [(\'image_full\', 1), (\'image\', -1)], \'correct\'),\n        (\'full_binding_pair\', \'binding_swap\', [(\'image_full\', 1), (\'image\', -1)], \'both_correct\'),\n    ]\n    comparisons = []\n    for name, scope, coefficients, metric in definitions:\n        source = pair_fam if metric == \'both_correct\' else (fam if scope == \'all\' else fam[fam.stratum == scope])\n        pivot = source.pivot(index=\'family_id\', columns=\'condition\', values=metric)\n        delta = sum(coef * pivot[condition] for condition, coef in coefficients)\n        comparisons.append(dict(model=model_key, phase=phase, comparison=name, metric=metric, scope=scope,\n                                **paired_stats(delta.to_numpy(), [model_key, phase, name], replicates)))\n    tests = pd.DataFrame(comparisons)\n    tests[\'holm_within_model\'] = holm(tests.p_value)\n    tests[\'inference_status\'] = \'development_descriptive_only\' if phase == \'development\' else \'pending_joint_14_test_correction\'\n    tests.to_csv(outdir / \'primary_comparisons.csv\', index=False)\n    event_rows = []\n    keys = [\'family_id\', \'scene_id\', \'role\', \'mirror\', \'mapping_index\']\n    base = df[df.condition == \'image\'][keys + [\'correct\']].rename(columns={\'correct\': \'baseline_correct\'})\n    for c in CONDITIONS:\n        joined = df[df.condition == c].merge(base, on=keys, validate=\'one_to_one\')\n        joined[\'repaired\'] = (joined.baseline_correct == 0) & (joined.correct == 1)\n        joined[\'harmed\'] = (joined.baseline_correct == 1) & (joined.correct == 0)\n        joined[\'original_error\'] = joined.baseline_correct == 0\n        joined[\'original_correct\'] = joined.baseline_correct == 1\n        for scope in (\'all\', *STRATA):\n            g = joined if scope == \'all\' else joined[joined.stratum == scope]\n            event_rows.append(dict(condition=c, scope=scope, n_trials=len(g),\n                                   repairs=int(g.repaired.sum()), harms=int(g.harmed.sum()),\n                                   original_errors=int(g.original_error.sum()), original_correct=int(g.original_correct.sum()),\n                                   repairs_per_original_error=float(g.repaired.sum()/g.original_error.sum()) if g.original_error.sum() else None,\n                                   harms_per_original_correct=float(g.harmed.sum()/g.original_correct.sum()) if g.original_correct.sum() else None))\n    pd.DataFrame(event_rows).to_csv(outdir / \'repair_harm_counts.csv\', index=False)\n    audits = df[df.generation_audited == True]\n    audit_summary = []\n    for condition, group in audits.groupby(\'condition\'):\n        valid = group.generated_label.isin(LABELS)\n        agreed = valid & (group.generated_label == group.predicted_label)\n        audit_summary.append(dict(condition=condition, n=len(group), valid=int(valid.sum()),\n                                  agrees=int(agreed.sum()), agreement_all=float(agreed.mean()),\n                                  agreement_valid=float(agreed.sum()/valid.sum()) if valid.sum() else None))\n    pd.DataFrame(audit_summary).to_csv(outdir / \'generation_scoring_agreement.csv\', index=False)\n    return summary, tests\n',"<embedded-parent>","exec"),_parent.__dict__)
exec(compile('"""CPU protocol, narrow response grammar and analysis for archived Gemma inputs."""\nfrom __future__ import annotations\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport re\nimport zipfile\nimport numpy as np\nimport pandas as pd\nfrom visual_information_support_core import digest, file_hash, read_json, save_json, freeze, paired_stats, holm\n\nSCHEMA = \'gemma-generation-recheck-v1\'\nMODEL_ID = \'google/gemma-4-12B-it\'\nREVISION = \'707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7\'\nSOURCE_NAME = \'VisualTorque_Frozen_Visual_Validation_gemma12b_Output.zip\'\nSOURCE_SHA = \'fcb26b5ef9d55ab1f79565bb4905f4a3f42d67a847d6c81d874edbfa0c47dffd\'\nSOURCE_IMPLEMENTATION = \'e84045fd42cd1f058593cdbd45e07115e0339b491ea91c315ffdee99be1e75d4\'\nPREFIX = \'models/gemma12b/frozen_validation/\'\nWORDINGS = (\'original\', \'neutral_question\')\nSTRATA = (\'congruent\', \'balanced\', \'equal_weight_distance\', \'weight_conflict\', \'binding_swap\')\nLABELS = (\'A\', \'B\', \'C\')\nPARSER_VERSION = \'complete-turn-single-letter-grammar-v1\'\nPARSER_RULES = {\n    \'strict\': \'Completed turn containing exactly A, B or C after whitespace trimming.\',\n    \'normalized\': \'Strict reply, or exactly thought + closing channel + letter, closing channel + letter, or a complete thought span followed by a single letter. All require <turn|> and no trailing content.\',\n    \'boundary\': \'Channel-wrapped outputs are explicitly marked; recovery defines a revised measurement and is not proof of a normal final channel or disabled internal reasoning.\',\n    \'invalid\': \'No completed turn, multiple letters, extra answer prose, unclosed/unknown/nested channel markup, and any ambiguous continuation remain invalid. Never search for the last letter.\',\n}\n\n\ndef parse_response(raw):\n    """Full-string grammar; do not erase arbitrary markup or search answer prose."""\n    text = raw.strip()\n    result = dict(strict_label=None, normalized_label=None, response_form=\'invalid\',\n                  reason=None, completed_turn=False, had_channel_wrapper=False)\n    if text.count(\'<turn|>\') != 1 or not text.endswith(\'<turn|>\'):\n        result[\'reason\'] = \'missing_or_ambiguous_turn_end\'\n        return result\n    body = text[:-len(\'<turn|>\')].strip()\n    result[\'completed_turn\'] = True\n    if body in LABELS:\n        return dict(result, strict_label=body, normalized_label=body, response_form=\'direct_letter\')\n    observed = re.fullmatch(r\'(thought\\s*)?<channel\\|>\\s*([ABC])\', body)\n    if observed:\n        return dict(result, normalized_label=observed.group(2), had_channel_wrapper=True,\n            response_form=\'observed_thought_wrapper\' if observed.group(1) else \'observed_closing_wrapper\')\n    structured = re.fullmatch(r\'<\\|channel>thought\\s+(.*?)<channel\\|>\\s*([ABC])\', body, flags=re.S)\n    if structured and \'<\' not in structured.group(1) and \'>\' not in structured.group(1):\n        return dict(result, normalized_label=structured.group(2), had_channel_wrapper=True,\n                    response_form=\'closed_thought_span\')\n    result[\'reason\'] = \'not_a_single_letter_under_fixed_grammar\'\n    return result\n\n\ndef choose_source(project_root, explicit=None):\n    candidates = [Path(explicit)] if explicit else [\n        Path(project_root)/\'results/frozen_visual_validation_v1/main\'/SOURCE_NAME,\n        Path(\'/content\')/SOURCE_NAME,\n        Path(project_root)/SOURCE_NAME,\n    ]\n    for path in candidates:\n        if path.is_file():\n            assert file_hash(path) == SOURCE_SHA, f\'Wrong source ZIP: {path}. Use the reviewed Gemma frozen-validation export.\'\n            return path\n    raise FileNotFoundError(\'Gemma source ZIP not found. Set SOURCE_ZIP to its exact Drive path, or upload \'+SOURCE_NAME+\' to /content. No recursive search is performed.\')\n\n\ndef prepare_source(path, shared_root):\n    """One named ZIP, verified in memory; no broad extraction or history scan."""\n    raw = Path(path).read_bytes()\n    assert hashlib.sha256(raw).hexdigest() == SOURCE_SHA\n    root = Path(shared_root); root.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(io.BytesIO(raw)) as z:\n        manifest = json.loads(z.read(\'export_manifest.json\'))\n        assert manifest[\'model\'] == \'gemma12b\' and manifest[\'status\'] == \'complete\'\n        assert len(z.namelist()) == len(set(z.namelist()))\n        assert set(z.namelist()) == set(manifest[\'files\']) | {\'export_manifest.json\'}\n        for name, h in manifest[\'files\'].items():\n            assert hashlib.sha256(z.read(name)).hexdigest() == h, name\n        runtime = json.loads(z.read(PREFIX+\'runtime_identity.json\'))\n        complete = json.loads(z.read(PREFIX+\'completion.json\'))\n        support = json.loads(z.read(\'shared/frozen_validation/support.json\'))\n        inputs = [x for x in json.loads(z.read(\'shared/frozen_validation/inputs.json\')) if x[\'condition\']==\'image\']\n        assert runtime[\'implementation\'] == SOURCE_IMPLEMENTATION\n        assert runtime[\'model_metadata\'][\'resolved_revision\'] == REVISION\n        assert complete[\'n_scored\'] == 8640 and len(support[\'families\']) == 96\n        assert len(inputs) == len({x[\'trial_id\'] for x in inputs}) == 1440\n        assert all(sum(x[\'wording\']==w for x in inputs)==720 for w in WORDINGS)\n        selected = {x[\'trial_id\'] for x in inputs}; old = {}\n        for name, h in complete[\'family_files\'].items():\n            payload_raw = z.read(PREFIX+\'families/\'+name)\n            assert hashlib.sha256(payload_raw).hexdigest() == h\n            payload = json.loads(payload_raw)\n            assert digest(payload[\'rows\']) == payload[\'rows_sha256\']\n            for row in payload[\'rows\']:\n                if row[\'trial_id\'] in selected:\n                    assert row[\'trial_id\'] not in old\n                    old[row[\'trial_id\']] = row\n        assert set(old) == selected\n        image_hashes={}\n        for x in inputs:\n            assert hashlib.sha256(x[\'prompt\'].encode()).hexdigest() == x[\'prompt_sha256\']\n            assert old[x[\'trial_id\']][\'prompt_sha256\'] == x[\'prompt_sha256\']\n            name=x[\'image_filename\']; assert Path(name).name == name\n            assert name not in image_hashes or image_hashes[name]==x[\'image_sha256\']\n            image_hashes[name]=x[\'image_sha256\']\n        for name,image_hash in image_hashes.items():\n            image_raw=z.read(\'shared/frozen_validation/images/\'+name)\n            assert hashlib.sha256(image_raw).hexdigest() == image_hash\n            dest=root/\'images\'/name; dest.parent.mkdir(exist_ok=True)\n            if dest.exists():\n                assert file_hash(dest)==image_hash\n            else:\n                dest.write_bytes(image_raw)\n        preflight=[x[\'trial_id\'] for x in inputs if x[\'generation_audit\']]\n        assert len(preflight)==40\n        snapshot=dict(source_zip_name=SOURCE_NAME,source_zip_sha256=SOURCE_SHA,\n            source_runtime=runtime,source_completion=complete,\n            source_protocol=json.loads(z.read(\'protocol.json\')),\n            scope=\'Post-hoc measurement recheck of 96 already observed physical families, not new held-out evidence.\')\n        for name,obj in [(\'source_snapshot.json\',snapshot),(\'support.json\',support),(\'inputs.json\',inputs),\n                         (\'archived_image_rows.json\',old),(\'preflight_ids.json\',preflight)]:\n            freeze(root/name,obj)\n    return snapshot,support,inputs,old,preflight\n\n\ndef validate_row(row, inp, runtime_sha):\n    assert row[\'runtime_sha256\']==runtime_sha and row[\'trial_id\']==inp[\'trial_id\']\n    assert row[\'prompt_sha256\']==inp[\'prompt_sha256\']\n    assert row[\'input_definition_sha256\']==digest(inp)\n    assert row[\'parsed\']==parse_response(row[\'raw_response\'])\n    assert len(row[\'generated_ids\'])==len(row[\'generated_token_pieces\'])==len(row[\'generation_steps\'])\n    for field in (\'family_id\',\'scene_id\',\'stratum\',\'role\',\'mirror\',\'mapping_index\',\'wording\',\'gold_label\',\'gold_semantic\',\'semantics\'):\n        assert row[field]==inp[field]\n\n\ndef load_rows(folder, inputs_by_id, runtime_sha):\n    result={}\n    for path in sorted(Path(folder).glob(\'*.json\')):\n        payload=read_json(path)\n        assert payload[\'runtime_sha256\']==runtime_sha\n        assert payload[\'rows_sha256\']==digest(payload[\'rows\'])\n        for row in payload[\'rows\']:\n            assert row[\'family_id\']==payload[\'family_id\']\n            assert row[\'trial_id\'] not in result\n            validate_row(row,inputs_by_id[row[\'trial_id\']],runtime_sha)\n            result[row[\'trial_id\']]=row\n    return result\n\n\ndef checkpoint(folder, family_id, rows, runtime_sha):\n    assert family_id and all(c.isalnum() or c in \'_-\' for c in family_id)\n    selected=sorted((r for r in rows.values() if r[\'family_id\']==family_id),key=lambda r:r[\'trial_id\'])\n    save_json(Path(folder)/(family_id+\'.json\'),dict(family_id=family_id,runtime_sha256=runtime_sha,\n        rows=selected,rows_sha256=digest(selected)))\n\n\ndef preflight_report(rows, ids, old, outdir):\n    assert set(ids)<=set(rows)\n    data=[];traces=[]\n    for tid in ids:\n        r=rows[tid];a=old[tid];p=r[\'parsed\']\n        before=np.array([a[\'candidate_logprobs\'][l] for l in LABELS])\n        after=np.array([r[\'replayed_candidate_logprobs\'][l] for l in LABELS])\n        data.append(dict(trial_id=tid,wording=r[\'wording\'],stratum=r[\'stratum\'],\n            raw_response=r[\'raw_response\'],response_form=p[\'response_form\'],normalized_label=p[\'normalized_label\'],\n            old_raw_response=a[\'raw_response\'],raw_exact_replay=r[\'raw_response\']==a[\'raw_response\'],\n            strict_label=p[\'strict_label\'],old_candidate_label=a[\'predicted_label\'],\n            input_identity_equal=r[\'prepared_inputs\']==a[\'prepared_inputs\'],\n            centered_candidate_error=float(np.max(np.abs((after-after.mean())-(before-before.mean())))),\n            first_step_candidate_mass=r[\'generation_steps\'][0][\'abc_probability_mass\'],\n            truncated=r[\'hit_token_limit_without_turn_end\']))\n        for index,(token,piece,step) in enumerate(zip(r[\'generated_ids\'],r[\'generated_token_pieces\'],r[\'generation_steps\'])):\n            traces.append(dict(trial_id=tid,wording=r[\'wording\'],step=index,token_id=token,\n                token_piece=piece,abc_probability_mass=step[\'abc_probability_mass\']))\n    df=pd.DataFrame(data);Path(outdir).mkdir(parents=True,exist_ok=True)\n    df.to_csv(Path(outdir)/\'preflight_comparison.csv\',index=False)\n    pd.DataFrame(traces).to_csv(Path(outdir)/\'preflight_token_trace.csv\',index=False)\n    # Gate checks technical interpretability only, never physical accuracy.\n    ready=bool(df.input_identity_equal.all() and df.normalized_label.notna().all() and not df.truncated.any())\n    report=dict(ready=ready,n=40,normalized_valid=int(df.normalized_label.notna().sum()),\n        strict_valid=int(df.strict_label.notna().sum()),raw_exact_replays=int(df.raw_exact_replay.sum()),\n        largest_centered_score_difference=float(df.centered_candidate_error.max()),\n        rows_with_score_difference_above_015=int((df.centered_candidate_error>.15).sum()),\n        gate=\'Exact input identity and complete, unambiguous response under the fixed grammar. No gate on physical correctness or agreement with archived candidate choices.\',\n        interpretation=\'Wrapper recovery is reported separately from strict direct letters; it does not establish normal channel behavior.\')\n    save_json(Path(outdir)/\'preflight_report.json\',report)\n    return report,df\n\n\ndef analyze(rows, inputs, old, outdir, replicates=10000):\n    outdir=Path(outdir);outdir.mkdir(parents=True,exist_ok=True)\n    assert len(rows)==len(inputs)==1440 and set(rows)=={x[\'trial_id\'] for x in inputs}\n    records=[]\n    for inp in inputs:\n        r=rows[inp[\'trial_id\']];p=r[\'parsed\']\n        record={k:inp[k] for k in (\'trial_id\',\'family_id\',\'stratum\',\'scene_id\',\'role\',\'mirror\',\'mapping_index\',\'wording\',\'gold_label\',\'gold_semantic\')}\n        for metric,label in [(\'normalized\',p[\'normalized_label\']),(\'strict\',p[\'strict_label\'])]:\n            sem=inp[\'semantics\'][LABELS.index(label)] if label in LABELS else None\n            record.update({metric+\'_valid\':int(label is not None),metric+\'_correct\':int(label==inp[\'gold_label\']),\n                metric+\'_balance\':int(sem==\'balanced\'),metric+\'_left\':int(sem==\'left\'),metric+\'_right\':int(sem==\'right\')})\n        record.update(response_form=p[\'response_form\'],normalized_label=p[\'normalized_label\'],strict_label=p[\'strict_label\'],\n            raw_response=r[\'raw_response\'],wrapper=int(p[\'had_channel_wrapper\']),\n            truncated=int(r[\'hit_token_limit_without_turn_end\']),archived_scored_correct=old[r[\'trial_id\']][\'correct\'],\n            archived_scored_label=old[r[\'trial_id\']][\'predicted_label\'],\n            generated_matches_archived_score=int(p[\'normalized_label\']==old[r[\'trial_id\']][\'predicted_label\']))\n        records.append(record)\n    df=pd.DataFrame(records);df.to_csv(outdir/\'generation_trials.csv\',index=False)\n    metrics=[c for c in df if c.startswith((\'normalized_\',\'strict_\')) and c not in (\'normalized_label\',\'strict_label\')]\n    metrics+=[\'wrapper\',\'truncated\',\'archived_scored_correct\',\'generated_matches_archived_score\']\n    fam=df.groupby([\'family_id\',\'stratum\',\'wording\'])[metrics].mean().reset_index()\n    fam.to_csv(outdir/\'family_metrics.csv\',index=False)\n    summary=[]\n    for scope in (\'all\',*STRATA):\n        g=fam if scope==\'all\' else fam[fam.stratum==scope]\n        for wording,group in g.groupby(\'wording\'):\n            for metric in metrics:\n                # Intervals are descriptive; do not test every table cell.\n                stats=paired_stats(group[metric].to_numpy(),[SCHEMA,scope,wording,metric],replicates)\n                stats.pop(\'p_value\')\n                summary.append(dict(scope=scope,wording=wording,metric=metric,**stats))\n    pd.DataFrame(summary).to_csv(outdir/\'accuracy_and_format_summary.csv\',index=False)\n    comparisons=[]\n    for name,scope,metric in [(\'overall_accuracy\',\'all\',\'normalized_correct\'),\n                              (\'true_balance_accuracy\',\'balanced\',\'normalized_correct\'),\n                              (\'false_balance_response\',\'nonbalanced\',\'normalized_balance\')]:\n        g=fam if scope==\'all\' else fam[fam.stratum==\'balanced\'] if scope==\'balanced\' else fam[fam.stratum!=\'balanced\']\n        pivot=g.pivot(index=\'family_id\',columns=\'wording\',values=metric)\n        comparisons.append(dict(comparison=name,scope=scope,metric=metric,\n            **paired_stats((pivot.neutral_question-pivot.original).to_numpy(),[SCHEMA,name],replicates)))\n    comp=pd.DataFrame(comparisons);comp[\'holm_recheck_3\']=holm(comp.p_value)\n    comp[\'status\']=\'post_hoc_measurement_recheck_same_families_not_fresh_confirmation\'\n    comp.to_csv(outdir/\'paired_wording_comparisons.csv\',index=False)\n    forms=df.groupby([\'wording\',\'response_form\']).size().reset_index(name=\'n\')\n    forms.to_csv(outdir/\'response_forms.csv\',index=False)\n    events=[]\n    for scope in (\'all\',*STRATA):\n        g=df if scope==\'all\' else df[df.stratum==scope]\n        pivot=g.pivot(index=[\'family_id\',\'scene_id\',\'mapping_index\'],columns=\'wording\',values=\'normalized_correct\')\n        errors=int((pivot.original==0).sum());correct=int((pivot.original==1).sum())\n        repair=int(((pivot.original==0)&(pivot.neutral_question==1)).sum())\n        harm=int(((pivot.original==1)&(pivot.neutral_question==0)).sum())\n        events.append(dict(scope=scope,n=len(pivot),repairs=repair,harms=harm,original_errors=errors,original_correct=correct,\n                           repair_rate=repair/errors if errors else None,harm_rate=harm/correct if correct else None))\n    pd.DataFrame(events).to_csv(outdir/\'repairs_and_harms.csv\',index=False)\n    save_json(outdir/\'measurement_status.json\',dict(n_trials=len(df),n_families=96,\n        normalized_invalid=int((1-df.normalized_valid).sum()),strict_invalid=int((1-df.strict_valid).sum()),\n        incomplete_turns=int(df.truncated.sum()),\n        normalized_interpretation=\'Single-letter generation after predefined response normalization; anomalous wrappers remain explicitly identified.\',\n        denominator=\'All 1440 trials; invalid replies count as unsuccessful, never silently removed. Family macro averaging.\',\n        multiplicity=\'Three revised Gemma wording comparisons form a separate post-hoc family; old 14/8-test results are preserved, not overwritten.\'))\n    return pd.DataFrame(summary),comp\n\n\ndef export_run(run_root, status):\n    root=Path(run_root);target=root/\'VisualTorque_Gemma_Generation_Recheck_Output.zip\'\n    files={}\n    for folder in (\'shared\',\'run\'):\n        for p in sorted((root/folder).rglob(\'*\')):\n            if p.is_file() and p.suffix!=\'.tmp\':files[p.relative_to(root).as_posix()]=file_hash(p)\n    for name in (\'protocol.json\',):\n        if (root/name).exists():files[name]=file_hash(root/name)\n    manifest=dict(schema=SCHEMA,status=status,model=\'gemma12b\',source_zip_sha256=SOURCE_SHA,files=files)\n    temp=target.with_suffix(\'.zip.tmp\')\n    with zipfile.ZipFile(temp,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for name in files:z.write(root/name,name)\n        z.writestr(\'export_manifest.json\',json.dumps(manifest,ensure_ascii=False,indent=2)+\'\\n\')\n    with zipfile.ZipFile(temp) as z:\n        assert z.testzip() is None\n        assert all(hashlib.sha256(z.read(n)).hexdigest()==h for n,h in files.items())\n    temp.replace(target)\n    return target\n',"<embedded-generation-recheck>","exec"),globals())
IMPLEMENTATION_SHA256='c064ade2c0fecc833cd55c88af0769bc53bf60b33eb1506ea3625a57e7e3570b'


In [ ]:
%%cell_progress 5/13 Freeze recheck protocol and source identity
assert RUN_ID and all(c.isalnum() or c in '_-' for c in RUN_ID)
RUN_ROOT=PROJECT_ROOT/'results/gemma_generation_recheck_v1'/RUN_ID
SHARED_ROOT=RUN_ROOT/'shared'
PHASE_ROOT=RUN_ROOT/'run'
ANALYSIS_DIR=PHASE_ROOT/'analysis'
FAMILY_ROOT=PHASE_ROOT/'families'
for folder in (SHARED_ROOT,ANALYSIS_DIR,FAMILY_ROOT):folder.mkdir(parents=True,exist_ok=True)
protocol=dict(schema=SCHEMA,implementation=IMPLEMENTATION_SHA256,model_id=MODEL_ID,revision=REVISION,
    source_name=SOURCE_NAME,source_zip_sha256=SOURCE_SHA,n_families=96,n_generations=1440,
    wordings=list(WORDINGS),condition='image',max_new_tokens=16,do_sample=False,batch_size=1,
    dtype='bfloat16',thinking_requested=False,parser_version=PARSER_VERSION,parser_rules=PARSER_RULES,
    preflight='The original image-condition audit IDs: 20 original plus their 20 neutral counterparts. Reuse these generated records in the full analysis.',
    gate='Identical archived input tensors; completed and unambiguous responses under the fixed grammar. No selection by physical correctness.',
    normalized_metric='Revised single-letter generation metric with the explicit fixed wrapper grammar; retain strict direct-letter accuracy and invalid rates beside it.',
    endpoints=['overall_accuracy','true_balance_accuracy','false_balance_response'],
    inference='Family-paired bootstrap intervals and sign-flip tests; Holm over three revised Gemma endpoints only. Post-hoc same-family recheck, not independent confirmation.',
    missing_answers='Keep every planned row. Invalid replies count as unsuccessful, remain their own category, and are never silently removed.',
    preservation='Read source ZIP only. No prompt/template/model revision/scoring overwrite; do not mix old next-letter scores with new generated choices.')
freeze(RUN_ROOT/'protocol.json',protocol)
SOURCE_PATH=choose_source(PROJECT_ROOT,SOURCE_ZIP)
print('Input ZIP:',SOURCE_PATH)
print('Separate output:',RUN_ROOT)


In [ ]:
%%cell_progress 6/13 Load generation instrumentation
"""Generation capture for one frozen Gemma protocol; embedded in the notebook."""
import gc
import shutil
import time
from datetime import datetime, timezone
from tqdm.auto import tqdm
from visual_torque_toolkit.behavior.data import BehaviorExample
from visual_torque_toolkit.behavior.schema import PromptCondition


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def tensor_identity(inputs):
    result={}
    for key,value in inputs.items():
        if torch.is_tensor(value):
            x=value.detach().cpu().contiguous()
            result[key]=dict(shape=list(x.shape),dtype=str(x.dtype),
                sha256=hashlib.sha256(x.view(torch.uint8).numpy().tobytes()).hexdigest())
        else:
            result[key]=str(value)
    return result


def example_for(inp):
    scene=SCENES[inp['scene_id']]
    semantics=tuple(inp['semantics'])
    words={'left':'left side','right':'right side','balanced':'equal'}
    if inp['wording']=='neutral_question':words['balanced']='balanced (neither side tips)'
    return BehaviorExample(example_id=inp['trial_id'],question_id=inp['trial_id'],scene_id=inp['scene_id'],
        condition=PromptCondition.IMAGE_RULE,question_kind='outcome',variant='image__'+inp['wording'],
        split='frozen_validation',problem_type=inp['stratum'],outcome=inp['gold_semantic'],
        net_torque=scene['features']['net_torque'],prompt=inp['prompt'],
        image_path=LOCAL_IMAGES/inp['image_filename'],labels=LABELS,
        option_text=tuple(words[s] for s in semantics),option_semantics=semantics,
        gold_label=inp['gold_label'],gold_semantic=inp['gold_semantic'])


def initialize_runtime():
    global RUNTIME_IDENTITY,RUNTIME_SHA
    import PIL,transformers
    assert adapter.metadata['resolved_revision']==REVISION
    assert adapter.metadata['dtype']=='bfloat16'
    assert not getattr(adapter.model,'is_quantized',False),'Use the native BF16 checkpoint.'
    assert not any(str(v) in ('cpu','disk') for v in getattr(adapter.model,'hf_device_map',{}).values()),'Full model must fit on GPU; no CPU/disk offload.'
    adapter.model.eval();torch.manual_seed(20260920);np.random.seed(20260920)
    torch.backends.cuda.matmul.allow_tf32=False
    processor=adapter.processor;tokenizer=processor.tokenizer
    serial=lambda obj:json.loads(json.dumps(obj,default=str))
    prior=SOURCE_SNAPSHOT['source_runtime']
    values=dict(model_metadata=adapter.metadata,gpu=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
        torch=torch.__version__,transformers=transformers.__version__,pillow=PIL.__version__,cuda=torch.version.cuda,
        numpy=np.__version__,pandas=pd.__version__,dtype='bfloat16',batch_size=1,thinking=False,
        toolkit={str(p.relative_to(TOOLKIT_ROOT)):file_hash(p) for p in sorted((TOOLKIT_ROOT/'src').rglob('*.py'))},
        processor_config=serial(processor.to_dict()),image_processor_config=serial(processor.image_processor.to_dict()),
        chat_template=processor.chat_template,tokenizer_chat_template=tokenizer.chat_template)
    differences={k:dict(identical=v==prior.get(k)) for k,v in values.items()}
    critical=('transformers','pillow','dtype','batch_size','thinking','toolkit','processor_config','image_processor_config','chat_template','tokenizer_chat_template')
    changed=[k for k in critical if not differences[k]['identical']]
    assert not changed,'Restore the archived toolkit/preprocessing identity before this recheck: '+str(changed)
    token_contract={}
    for text in ('<|turn>','<turn|>','<|channel>','<channel|>','A','B','C'):
        ids=tokenizer(text,add_special_tokens=False)['input_ids']
        assert len(ids)==1,('Expected pinned single-token contract',text,ids)
        decoded=processor.decode(ids,skip_special_tokens=False,clean_up_tokenization_spaces=False)
        assert decoded==text,(text,decoded)
        token_contract[text]=dict(ids=ids,special=ids[0] in tokenizer.all_special_ids,decoded=decoded)
    assert token_contract['<turn|>']['special']
    eos=adapter.model.generation_config.eos_token_id
    eos=[eos] if isinstance(eos,int) else list(eos or [])
    assert token_contract['<turn|>']['ids'][0] in eos,'The archived turn-end token is not an active generation stop.'
    plans={}
    for wording in WORDINGS:
        ex=example_for(next(i for i in INPUTS if i['wording']==wording))
        plan=adapter._candidate_plan(ex)
        assert plan is not None and not plan[0]
        plans[wording]=dict(prefix=list(plan[0]),token_by_label=plan[1])
        assert plans[wording]==prior['candidate_plans']['image']
    values.update(schema=SCHEMA,implementation=IMPLEMENTATION_SHA256,parser_version=PARSER_VERSION,
        source_zip_sha256=SOURCE_SHA,protocol_sha256=file_hash(RUN_ROOT/'protocol.json'),
        inputs_sha256=file_hash(SHARED_ROOT/'inputs.json'),generation_config=serial(adapter.model.generation_config.to_dict()),
        generation_overrides=dict(do_sample=False,max_new_tokens=16,pad_token_id=tokenizer.pad_token_id),
        attention_implementation=serial(getattr(adapter.model.config,'_attn_implementation',None)),
        token_contract=token_contract,candidate_plans=plans)
    RUNTIME_IDENTITY=values;RUNTIME_SHA=freeze(PHASE_ROOT/'runtime_identity.json',values)
    freeze(PHASE_ROOT/'source_environment_comparison.json',differences)
    print('Changed recorded environment fields:',[k for k,v in differences.items() if not v['identical']] or 'none')
    print('Response grammar frozen before generation. Channel wrappers remain separately labeled.')
    display(pd.DataFrame([dict(text=k,**v) for k,v in token_contract.items()]))


def capture(inp, *, preflight=False):
    ex=example_for(inp)
    inputs=adapter._to_device(adapter._template([ex.messages()],add_generation_prompt=True))
    identity=tensor_identity(inputs);old=ARCHIVED_ROWS[inp['trial_id']]
    assert identity==old['prepared_inputs'],'Input tensors differ from archive: '+inp['trial_id']
    assert inputs['input_ids'].shape[0]==1
    prompt_ids=inputs['input_ids'][0].detach().cpu().tolist()
    assert prompt_ids==old['input_token_ids']
    plan=adapter._candidate_plan(ex);assert plan is not None and not plan[0]
    tokens=plan[1];assert tokens==old['candidate_tokens']
    replay_scores=None
    if preflight:
        with torch.inference_mode():
            logits=adapter.model(**dict(inputs),use_cache=False).logits[0,-1].float()
            lp=torch.log_softmax(logits,dim=-1)
            replay_scores={l:float(lp[tokens[l]].item()) for l in LABELS}
        assert np.isfinite(list(replay_scores.values())).all()
        del logits,lp
    started=time.monotonic();width=inputs['input_ids'].shape[1]
    # Same decoding arguments as the archived adapter; return scores only adds instrumentation.
    with torch.inference_mode():
        output=adapter.model.generate(**dict(inputs),do_sample=False,max_new_tokens=16,
            pad_token_id=adapter.processor.tokenizer.pad_token_id,return_dict_in_generate=True,output_scores=True)
    ids=output.sequences[0,width:].detach().cpu().tolist()
    assert ids and len(ids)==len(output.scores)
    raw=adapter.processor.decode(ids,skip_special_tokens=False,clean_up_tokenization_spaces=False).strip()
    parsed=parse_response(raw)
    steps=[]
    finite=lambda value:float(value) if np.isfinite(float(value)) else None
    for index,(token,score) in enumerate(zip(ids,output.scores)):
        lp=torch.log_softmax(score[0].float(),dim=-1)
        candidate={l:finite(lp[tokens[l]].item()) for l in LABELS}
        steps.append(dict(index=index,token_id=int(token),token_logprob=finite(lp[token].item()),
            argmax_token_id=int(score[0].argmax().item()),candidate_logprobs=candidate,
            abc_probability_mass=float(torch.exp(lp[[tokens[l] for l in LABELS]]).sum().item())))
    inp_meta={k:inp[k] for k in ('trial_id','family_id','scene_id','stratum','role','mirror','mapping_index','wording','gold_label','gold_semantic','semantics','prompt_sha256')}
    row=dict(inp_meta,runtime_sha256=RUNTIME_SHA,input_definition_sha256=digest(inp),
        prepared_inputs=identity,input_sha256=digest(identity),input_token_ids=prompt_ids,
        prompt_tail_decoded=adapter.processor.decode(prompt_ids[-64:],skip_special_tokens=False,clean_up_tokenization_spaces=False),
        generated_ids=ids,generated_token_pieces=[adapter.processor.tokenizer.convert_ids_to_tokens(i) for i in ids],
        generation_steps=steps,raw_response=raw,parsed=parsed,
        archived_adapter_parsed=adapter._parse_generated_content(raw,inputs['input_ids'][0],output.sequences[0,width:]),
        hit_token_limit_without_turn_end=len(ids)>=16 and not parsed['completed_turn'],
        replayed_candidate_logprobs=replay_scores,preflight=preflight,
        generation_seconds=time.monotonic()-started,recorded_at_utc=now_utc())
    validate_row(row,inp,RUNTIME_SHA)
    del output,inputs,lp
    return row


def run_preflight():
    rows=load_rows(FAMILY_ROOT,INPUT_BY_ID,RUNTIME_SHA)
    try:
        for tid in tqdm(PREFLIGHT_IDS,desc='40 paired boundary checks',unit='input'):
            if tid not in rows:
                rows[tid]=capture(INPUT_BY_ID[tid],preflight=True)
                checkpoint(FAMILY_ROOT,rows[tid]['family_id'],rows,RUNTIME_SHA)
            assert rows[tid]['replayed_candidate_logprobs'] is not None
        report,table=preflight_report(rows,PREFLIGHT_IDS,ARCHIVED_ROWS,ANALYSIS_DIR)
        display(table[['wording','stratum','raw_response','response_form','normalized_label','old_candidate_label','raw_exact_replay']])
        print(json.dumps(report,ensure_ascii=False,indent=2))
        print('Example exact prompt ending:',repr(rows[PREFLIGHT_IDS[0]]['prompt_tail_decoded']))
        trace=pd.read_csv(ANALYSIS_DIR/'preflight_token_trace.csv')
        example_ids=[next(t for t in PREFLIGHT_IDS if INPUT_BY_ID[t]['wording']==w) for w in WORDINGS]
        display(trace[trace.trial_id.isin(example_ids)])
        assert report['ready'],'An unrecognized/incomplete response or input mismatch needs inspection. Full generation is stopped; partial output has been saved. Do not relax parsing or select cases by accuracy.'
        freeze(PHASE_ROOT/'preflight_pass.json',dict(runtime_sha256=RUNTIME_SHA,ids=PREFLIGHT_IDS,
            report_sha256=file_hash(ANALYSIS_DIR/'preflight_report.json'),passed_at_rows_sha256=digest([rows[t] for t in PREFLIGHT_IDS])))
    except BaseException:
        path=export_run(RUN_ROOT,'preflight_incomplete_or_needs_review')
        print('Partial evidence ZIP:',path)
        raise
    return rows


def run_full():
    gate=read_json(PHASE_ROOT/'preflight_pass.json');assert gate['runtime_sha256']==RUNTIME_SHA
    rows=load_rows(FAMILY_ROOT,INPUT_BY_ID,RUNTIME_SHA)
    assert digest([rows[t] for t in PREFLIGHT_IDS])==gate['passed_at_rows_sha256']
    current_family=None
    try:
        for inp in tqdm(INPUTS,desc='1440 paired generations (cached rows reused)',unit='input'):
            if inp['trial_id'] in rows:continue
            if current_family is not None and current_family!=inp['family_id']:
                checkpoint(FAMILY_ROOT,current_family,rows,RUNTIME_SHA)
            current_family=inp['family_id']
            rows[inp['trial_id']]=capture(inp)
        if current_family is not None:checkpoint(FAMILY_ROOT,current_family,rows,RUNTIME_SHA)
    except BaseException:
        if current_family is not None:checkpoint(FAMILY_ROOT,current_family,rows,RUNTIME_SHA)
        print('Partial evidence ZIP:',export_run(RUN_ROOT,'partial'))
        raise
    assert len(rows)==1440
    return rows


def finish(rows):
    summary,comparisons=analyze(rows,INPUTS,ARCHIVED_ROWS,ANALYSIS_DIR)
    checks=read_json(ANALYSIS_DIR/'measurement_status.json')
    status='complete' if checks['normalized_invalid']==0 else 'complete_with_unresolved_responses'
    save_json(PHASE_ROOT/'completion.json',dict(status=status,n_generated=len(rows),n_families=96,
        runtime_sha256=RUNTIME_SHA,implementation=IMPLEMENTATION_SHA256,source_zip_sha256=SOURCE_SHA,
        completed_at_utc=now_utc(),family_files={p.name:file_hash(p) for p in sorted(FAMILY_ROOT.glob('*.json'))},
        analysis_files={p.name:file_hash(p) for p in sorted(ANALYSIS_DIR.glob('*')) if p.is_file()},
        meaning='Generation acquisition and analysis complete. Normalized wrappers are not ordinary direct letters; consult measurement_status and both metrics.'))
    display(summary[(summary.scope=='all') & summary.metric.isin(['normalized_correct','strict_correct','normalized_valid','strict_valid','wrapper'])].round(4))
    display(comparisons.round(4));display(pd.read_csv(ANALYSIS_DIR/'response_forms.csv'))
    print(json.dumps(checks,ensure_ascii=False,indent=2))
    return status


In [ ]:
%%cell_progress 7/13 Load archived images and paired inputs
cell_stage('Reading one verified ZIP; reusing the frozen scenes and exact prompts')
SOURCE_SNAPSHOT,SUPPORT,INPUTS,ARCHIVED_ROWS,PREFLIGHT_IDS=prepare_source(SOURCE_PATH,SHARED_ROOT)
INPUT_BY_ID={x['trial_id']:x for x in INPUTS}
SCENES={s['scene_id']:s for f in SUPPORT['families'] for s in f['scenes']}
# Copy only the 240 required pictures to local runtime storage for inference.
LOCAL_IMAGES=Path('/content/visualtorque_gemma_generation_recheck')/RUN_ID/'images'
LOCAL_IMAGES.mkdir(parents=True,exist_ok=True)
for inp in {x['image_filename']:x for x in INPUTS}.values():
    target=LOCAL_IMAGES/inp['image_filename']
    if not target.is_file() or file_hash(target)!=inp['image_sha256']:
        shutil.copyfile(SHARED_ROOT/'images'/inp['image_filename'],target)
    assert file_hash(target)==inp['image_sha256']
print('96 existing families; 720 original + 720 neutral inputs. No scene generation or Drive scan.')
print('40 preflight generations are included in, not added to, the 1440 total.')


In [ ]:
%%cell_progress 8/13 Check GPU and model access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Use an A100 40GB/80GB or a GPU that holds Gemma-12B in full BF16.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login unavailable:', type(exc).__name__)


In [ ]:
%%cell_progress 9/13 Load Gemma and verify runtime identity
cell_stage('Loading Gemma-12B with the archived revision in native BF16')
if 'adapter' in globals():
    del adapter
    gc.collect()
    torch.cuda.empty_cache()
adapter=HuggingFaceMultimodalAdapter(MODEL_ID,revision=REVISION,dtype='bfloat16',device_map='auto')
display(adapter.metadata)
initialize_runtime()


In [ ]:
%%cell_progress 10/13 Inspect 40 archived paired generations
PREFLIGHT_ROWS=run_preflight()


In [ ]:
%%cell_progress 11/13 Complete 1440 actual generations
ROWS=run_full()


In [ ]:
%%cell_progress 12/13 Compare both wordings with the same generation metric
STATUS=finish(ROWS)


In [ ]:
%%cell_progress 13/13 Export complete or partial evidence
completion=read_json(PHASE_ROOT/'completion.json') if (PHASE_ROOT/'completion.json').is_file() else {}
status=completion.get('status','partial')
OUTPUT_ZIP=export_run(RUN_ROOT,status)
print('Status:',status)
print('Please return this ZIP:',OUTPUT_ZIP)
print('Original frozen-validation ZIP and results were not modified.')
LOCAL_OUTPUT=Path('/content')/OUTPUT_ZIP.name
shutil.copyfile(OUTPUT_ZIP,LOCAL_OUTPUT)
if DOWNLOAD_OUTPUT:
    try:
        from google.colab import files
        files.download(str(LOCAL_OUTPUT))
    except Exception as exc:
        print('Browser download unavailable:',type(exc).__name__,'; use the saved Drive ZIP.')
print('Runtime-local copy:',LOCAL_OUTPUT)
